In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import scanpy as sc
import anndata

import glob

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import to_rgb

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/hicluster'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
log2_min = np.log2(2500)        # ≈ 11.29
log2_max = np.log2(250000000) # ≈ 27.89
step = 0.125
log2_bins = np.arange(log2_min, log2_max + step, step)

bin_edges = 2 ** log2_bins  # convert back to basepair units
bin_edges.shape

In [ ]:
def format_bp(bp):
    if bp < 1e3:
        return f"{bp}bp"
    elif bp < 1e6:
        return f"{bp/1e3:.1f} KB"
    else:
        return f"{bp/1e6:.1f} MB"

In [ ]:
vals = [int(x) for x in bin_edges[1:]]
tmp_cols = labels = [format_bp(v) for v in vals]
tmp_cols[:5]

In [ ]:
contact_df = pd.read_csv(f'{projdir}/csv/distance/merged.csv.gz', sep='\t', header=0, index_col=0)
contact_df.shape

In [ ]:
contact_df.head()

In [ ]:
contact_df.columns = tmp_cols

In [ ]:
contact_zscore = (contact_df.sub(contact_df.mean(axis=1), axis=0).div(contact_df.std(axis=1), axis=0))
contact_zscore.head()

In [ ]:
contact_zscore.min(axis=1), contact_zscore.max(axis=1)

In [ ]:
# contact_zscore.max(), contact_zscore.min()

In [ ]:
to_plot = contact_df.T
fig, ax = plt.subplots(figsize=(30, 10))

ax = sns.heatmap(to_plot, vmin=0, vmax=0.02, cmap='bwr')
ax.set_xticklabels([]);
# ax.set_yticklabels([]);

In [ ]:
# to_plot = contact_df.T
# fig, ax = plt.subplots(figsize=(30, 10))

# ax = sns.heatmap(to_plot, vmin=0, vmax=0.02, cmap='bwr')
# ax.set_xticklabels([]);
# ax.set_yticklabels([]);

In [ ]:
to_plot = contact_zscore.T
fig, ax = plt.subplots(figsize=(30, 10))

ax = sns.heatmap(to_plot, vmin=-2, vmax=2, cmap='bwr')
ax.set_xticklabels([]);
ax.set_yticklabels([]);

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc'
# clusters = pd.read_csv(f'{indir}/csv/label_transfer/justin_liftover_subclusters.csv', sep='\t', index_col=0)
clusters = pd.read_csv(f'{indir}/csv/label_transfer/xgboost_time_v7.csv', sep='\t', index_col=0)
clusters.shape

In [ ]:
clusters.head()

In [ ]:
sorted_clusters = clusters.sort_values(by=['cluster', 'line'])
cluster_nuclei = sorted_clusters.index
zscore_reorder = contact_zscore.reindex(cluster_nuclei).dropna()
zscore_reorder.shape

In [ ]:
zscore_reorder.head()

In [ ]:
labels = sorted_clusters['cluster'].values
change_indices = np.where(labels[1:] != labels[:-1])[0] + 1  # add 1 for boundary index
change_indices

In [ ]:
sorted_clusters['cluster'].unique()

In [ ]:
to_plot = zscore_reorder.T
fig, ax = plt.subplots(figsize=(30, 10))

ax = sns.heatmap(to_plot, vmin=-2, vmax=4, cmap='coolwarm')

for idx in change_indices:
    ax.axvline(idx, color='black', linewidth=1)
    
ax.set_xticklabels([]);
ax.set_yticklabels([]);

### add cluster labels

In [ ]:
to_plot = contact_zscore.T

In [ ]:
final_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1', 'Inter2', 'Fail2', 'IPS']

In [ ]:
results = []
for clust in final_order:
    nuclei = sorted_clusters[sorted_clusters['cluster'] == clust].index
    results.append(contact_zscore.loc[nuclei])
    
zscore_reorder = pd.concat(results)
zscore_reorder.shape

In [ ]:
results = []
for clust in final_order:
    nuclei = sorted_clusters[sorted_clusters['cluster'] == clust].index
    results.append(sorted_clusters.loc[nuclei])
    
reorder_clusters = pd.concat(results)
reorder_clusters.shape
reorder_clusters.shape

In [ ]:
cluster_labels = reorder_clusters['cluster']
unique_clusters = list(dict.fromkeys(cluster_labels))
tab10_colors = plt.get_cmap('tab10').colors
cluster_palette = {
    cluster: tab10_colors[i % len(tab10_colors)]
    for i, cluster in enumerate(unique_clusters)
}
cluster_colors = [cluster_palette[x] for x in cluster_labels]
len(cluster_colors)

In [ ]:
# reorder_clusters.to_csv('c')

In [ ]:
###
to_plot = zscore_reorder.T

fig = plt.figure(figsize=(20, 10))
gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[0.3, 6.0], hspace=0.05)

# Top axis: cluster color bar
ax_bar = fig.add_subplot(gs[0])
ax_bar.imshow([cluster_colors], aspect='auto')
ax_bar.set_xticks([])
ax_bar.set_yticks([])
ax_bar.set_ylabel('Cluster', fontsize=10, rotation=0, labelpad=20)
ax_bar.yaxis.set_label_position("left")
ax_bar.spines[:].set_visible(False)

# Bottom axis: heatmap
ax_heatmap = fig.add_subplot(gs[1])
sns.heatmap(to_plot, ax=ax_heatmap, cmap='coolwarm', vmin=-2, vmax=4, cbar=False,
            xticklabels=False, yticklabels=False)

# Legend below
handles = [
    mpatches.Patch(color=color, label=label)
    for label, color in cluster_palette.items()
]

fig.legend(
    handles=handles,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.05),
    ncol=len(handles),
    frameon=False
)

plt.tight_layout()
plt.show()

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
to_plot.to_csv(f'{outdir}/hic_contact_dist.csv', sep='\t')
reorder_clusters.to_csv(f'{outdir}/hic_contact_dist_meta.csv', sep='\t')

### plot it per donor and sort by short/long ratio increasing within each cluster

In [ ]:
contact_df = pd.read_csv(f'{projdir}/csv/distance/merged.csv.gz', sep='\t', header=0, index_col=0)
contact_df.columns = bin_edges[1:]

In [ ]:
cutoff = 2e6
short_bins = bin_edges[bin_edges < cutoff][1:]
long_bins = bin_edges[bin_edges > cutoff]

In [ ]:
short_sum = contact_df.loc[:,short_bins].sum(axis=1)
long_sum = contact_df.loc[:,long_bins].sum(axis=1)
sum_df = pd.DataFrame(zip(short_sum, long_sum))
sum_df.columns = ['short', 'long']
sum_df['short/long'] = sum_df['short'] / sum_df['long']
sum_df.index = contact_df.index
sum_df.shape

In [ ]:
sum_df.head()

In [ ]:
sum_df['donor'] = [x.split('-')[1][1:4] for x in sum_df.index]
sum_df['donor'] = sum_df['donor'].replace({'39D' : 'C39'})
sum_df['donor'].value_counts()

In [ ]:
sum_df['cluster'] = clusters.reindex(sum_df.index)['cluster']
sum_df = sum_df[~sum_df['cluster'].isna()]
sum_df.shape

In [ ]:
sum_df

In [ ]:
clusters.reindex(to_plot.columns[:-1])

In [ ]:
contacts_list = []
meta_list = []
for d in donors:
    donor_bcs = []
    tmp_df = sum_df[sum_df['donor'] == d]
    for s in tmp_order:
        final_df = tmp_df[tmp_df['cluster'] == s]
#         tmp_bcs = pd.DataFrame(final_df.sort_values(by=['short/long'], ascending=False).index)
        tmp_bcs = pd.DataFrame(final_df.sort_values(by=['short/long'], ascending=True).index)
        donor_bcs.append(tmp_bcs)
    donor_bcs = pd.concat(donor_bcs)[0].values

    to_plot = contact_zscore.reindex(donor_bcs).T

    fig = plt.figure(figsize=(20, 5))
    ax = sns.heatmap(to_plot, cmap='coolwarm', vmin=-2, vmax=4)

    ax.set_xticklabels([]);
    ax.set_yticklabels([]);
    
#     to_plot['donor'] = d
    contacts_list.append(to_plot)

    ###
    tmp_clusters = clusters.reindex(to_plot.columns[:-1])
    meta_list.append(tmp_clusters)
    
    print(to_plot.shape, tmp_clusters.shape)

In [ ]:
merged_meta = pd.concat(meta_list)
merged_contacts = pd.concat(contacts_list, axis=1)
merged_contacts = merged_contacts.T.reindex(merged_meta.index).T
merged_meta.shape, merged_contacts.shape

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
merged_contacts.to_csv(f'{outdir}/hic_contact_dist_sorted.csv.gz', sep='\t')
merged_meta.to_csv(f'{outdir}/hic_contact_dist_meta_sorted.csv.gz', sep='\t')

In [ ]:
!date